# SurfaceTarget-CRC — phase 05 — patch 5 5

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 5 — PATCH 5.5
#
# RANKING ROBUSTNESS AND CANDIDATE EVIDENCE DOSSIERS
#
# Objectives:
# 1. Validate Patch 5.4 ranking results
# 2. Audit correlations between ranking scenarios
# 3. Compare top-10, top-20 and top-30 membership
# 4. Distinguish 2-scenario from 3-scenario agreement
# 5. Audit normal-tissue IHC across all available tissues
# 6. Prepare candidate-specific evidence dossiers
# 7. Identify experimental validation priorities
# 8. Preserve original 948-gene research universe
# 9. Save diagnostic results and provenance
# 10. Create a research checkpoint
#
# IMPORTANT:
# - No therapeutic efficacy scores
# - No clinical safety predictions
# - No additional ranking weights
# - No source data overwritten
# - No candidates excluded
# - No malignant-cell localization inferred from bulk data
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED

import pandas as pd
import numpy as np
import hashlib
import json


# ------------------------------------------------------------
# 1. PROJECT CONFIGURATION
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

RESULTS = ROOT / "results/tables"
META = ROOT / "data/metadata"
CHECKPOINTS = ROOT / "checkpoints"

for folder in [RESULTS, META, CHECKPOINTS]:
    folder.mkdir(parents=True, exist_ok=True)


RANKING_PATH = (
    RESULTS /
    "surfacetarget_phase5_prioritization_v1.csv"
)

PROVENANCE_PATH = (
    META /
    "surfacetarget_phase5_prioritization_provenance.json"
)

TISSUE_PATH = (
    RESULTS /
    "hpa_candidate_tissue_level_profiles.csv"
)


SURFACE_LANE = "Putative cell-surface antigen review"

STROMAL_LANE = "Stromal-associated targeting review"

MEMBRANE_LANE = (
    "Membrane localization/accessibility verification"
)

EXTRACELLULAR_LANE = (
    "Extracellular microenvironment review"
)


SCENARIO_COLUMNS = {
    "protein_first": "phase5_rank_protein_first",
    "normal_review_first": "phase5_rank_normal_review_first",
    "rna_discovery_first": "phase5_rank_rna_discovery_first"
}


REVIEW_GENES = [
    "CA9",
    "ADAM12",
    "NPC1L1",
    "LGR5",
    "CDH3",
    "CLDN1",
    "FAP",
    "DPEP1",
    "SLC7A5",
    "SLC5A6",
    "TDGF1",
    "CEMIP",
    "CTHRC1",
    "MXRA5",
    "SFRP4"
]


print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 5 — PATCH 5.5")
print("RANKING ROBUSTNESS AND EVIDENCE DOSSIERS")
print("=" * 65)


# ------------------------------------------------------------
# 2. SOURCE INTEGRITY VALIDATION
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING RESEARCH INPUTS")
print("-" * 65)


for path in [
    RANKING_PATH,
    PROVENANCE_PATH,
    TISSUE_PATH
]:
    if not path.exists():
        raise FileNotFoundError(path)


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


with open(PROVENANCE_PATH, "r") as handle:
    provenance = json.load(handle)


expected_hash = provenance.get("output_sha256")
actual_hash = sha256_file(RANKING_PATH)

if not expected_hash:
    raise RuntimeError(
        "Missing Patch 5.4 output checksum."
    )

if expected_hash != actual_hash:
    raise RuntimeError(
        "Patch 5.4 ranking checksum mismatch."
    )


print("\nRanking source checksum: VERIFIED")


# ------------------------------------------------------------
# 3. LOAD CANONICAL RANKING DATA
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING EXISTING CANDIDATE PRIORITIES")
print("-" * 65)


source = pd.read_csv(
    RANKING_PATH,
    dtype={
        "gene_id": str,
        "ensembl_stable_id": str
    },
    low_memory=False
)


if len(source) != 948:
    raise RuntimeError(
        "Expected 948 research candidates."
    )

if source["gene_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate research gene identifiers."
    )

if source["ensembl_stable_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate Ensembl stable identifiers."
    )


required = {
    "gene_id",
    "ensembl_stable_id",
    "gene_name",
    "phase5_followup_lane",
    "phase5_topology_class",
    "phase5_protein_evidence_state",
    "phase5_normal_tissue_review_state",
    "phase5_normal_ihc_available",
    "phase5_rna_conservative_log2fc",
    "phase5_median_scenario_rank",
    "phase5_surface_consensus_position",
    "baseline_log2fc",
    "sensitivity_log2fc",
    "mss_complete_pairs",
    "protein_enriched",
    "depmap_crc_median_gene_effect",
    *SCENARIO_COLUMNS.values()
}


missing = required - set(source.columns)

if missing:
    raise RuntimeError(
        f"Missing required fields: {sorted(missing)}"
    )


surface = source[
    source["phase5_followup_lane"].eq(SURFACE_LANE)
].copy()


if len(surface) != 111:
    raise RuntimeError(
        "Expected 111 surface-review candidates."
    )


for column in SCENARIO_COLUMNS.values():

    ranks = pd.to_numeric(
        surface[column],
        errors="raise"
    ).astype(int)

    if set(ranks) != set(range(1, 112)):
        raise RuntimeError(
            f"Invalid scenario rank inventory: {column}"
        )


print("\nAll research candidates:", len(source))
print("Surface-review candidates:", len(surface))
print("Ranking scenarios:", len(SCENARIO_COLUMNS))
print("Ranking source validation: PASSED")


# ------------------------------------------------------------
# 4. RANK CORRELATION DIAGNOSTICS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CALCULATING SCENARIO RANK CORRELATIONS")
print("-" * 65)


rank_data = surface[
    list(SCENARIO_COLUMNS.values())
].copy()


rank_correlations = rank_data.corr(
    method="spearman"
)


rank_correlations.index = list(
    SCENARIO_COLUMNS.keys()
)

rank_correlations.columns = list(
    SCENARIO_COLUMNS.keys()
)


print("\nSpearman rank correlations:")

print(
    rank_correlations.to_string(
        float_format=lambda x: f"{x:.4f}"
    )
)


print(
    "\nThese correlations describe agreement "
    "between selected prioritization rules."
)

print(
    "They do not establish biological "
    "or therapeutic validation."
)


# ------------------------------------------------------------
# 5. TOP-N SCENARIO OVERLAP
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("AUDITING TOP-N RANKING OVERLAP")
print("-" * 65)


TOP_N_VALUES = [10, 20, 30]

overlap_records = []
consensus_records = []

scenario_names = list(
    SCENARIO_COLUMNS.keys()
)


for top_n in TOP_N_VALUES:

    top_sets = {}

    for name, column in SCENARIO_COLUMNS.items():

        top_sets[name] = set(
            surface.loc[
                surface[column] <= top_n,
                "gene_id"
            ]
        )

    for i in range(len(scenario_names)):

        for j in range(i + 1, len(scenario_names)):

            a = scenario_names[i]
            b = scenario_names[j]

            intersection = (
                top_sets[a] & top_sets[b]
            )

            union = (
                top_sets[a] | top_sets[b]
            )

            overlap_records.append({

                "top_n": top_n,
                "scenario_a": a,
                "scenario_b": b,
                "shared_candidates": len(intersection),
                "jaccard_similarity": (
                    len(intersection) / len(union)
                    if union
                    else np.nan
                )
            })


    # Distinguish agreement under all three
    # scenarios from agreement under only two.

    all_three = set.intersection(
        *top_sets.values()
    )

    at_least_two = {
        gene_id
        for gene_id in surface["gene_id"]
        if sum(
            gene_id in selected
            for selected in top_sets.values()
        ) >= 2
    }

    consensus_records.append({

        "top_n": top_n,

        "all_three_scenarios":
            len(all_three),

        "at_least_two_scenarios":
            len(at_least_two),

        "two_but_not_three":
            len(at_least_two - all_three)
    })


overlap = pd.DataFrame(overlap_records)

consensus = pd.DataFrame(consensus_records)


print("\nPairwise overlap:")

print(
    overlap.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)


print("\nAgreement across scenarios:")

print(
    consensus.to_string(index=False)
)


print("\nInterpretation:")

print(
    "Agreement under two strongly correlated "
    "scenarios is not equivalent to agreement "
    "under three independent evidence assumptions."
)


# ------------------------------------------------------------
# 6. IDENTIFY ALL-THREE TOP-20 CANDIDATES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("IDENTIFYING THREE-SCENARIO AGREEMENT")
print("-" * 65)


three_way_mask = np.logical_and.reduce([

    surface[column].le(20).to_numpy()

    for column in SCENARIO_COLUMNS.values()

])


three_way = surface.loc[
    three_way_mask
].copy()


three_way = three_way.sort_values(
    "phase5_median_scenario_rank"
)


print("\nCandidates appearing in all three top-20 lists:")

if not three_way.empty:

    print(
        three_way[
            [
                "gene_name",
                *SCENARIO_COLUMNS.values(),
                "protein_enriched",
                "phase5_normal_tissue_review_state"
            ]
        ].to_string(index=False)
    )

else:

    print("None.")


print(
    "\nThree-scenario agreement remains "
    "a ranking-consistency observation, "
    "not experimental confirmation."
)


# ------------------------------------------------------------
# 7. LOAD ALL HPA NORMAL-TISSUE PROFILES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("AUDITING ALL AVAILABLE NORMAL TISSUES")
print("-" * 65)


tissues = pd.read_csv(
    TISSUE_PATH,
    dtype={
        "Gene": str,
        "Tissue": str
    },
    low_memory=False
)


required_tissue_columns = {
    "Gene",
    "Tissue",
    "max_level_score",
    "max_reliable_score"
}


if not required_tissue_columns.issubset(
    tissues.columns
):

    raise RuntimeError(
        "Unexpected HPA tissue-profile schema."
    )


if tissues[
    ["Gene", "Tissue"]
].duplicated().any():

    raise RuntimeError(
        "Duplicate HPA gene-tissue profiles."
    )


if not set(tissues["Gene"]).issubset(
    set(source["ensembl_stable_id"])
):

    raise RuntimeError(
        "Unexpected Ensembl gene identifiers "
        "in HPA tissue profiles."
    )


for column in [
    "max_level_score",
    "max_reliable_score"
]:

    tissues[column] = pd.to_numeric(
        tissues[column],
        errors="coerce"
    )


print("\nGene-tissue profiles:", len(tissues))
print("Normal tissues:", tissues["Tissue"].nunique())
print("Genes represented:", tissues["Gene"].nunique())


# ------------------------------------------------------------
# 8. SUMMARIZE NORMAL-TISSUE EXPRESSION
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING ALL-TISSUE IHC SUMMARIES")
print("-" * 65)


tissue_records = []


for ensembl_id, group in tissues.groupby(
    "Gene",
    sort=False
):

    measured = group[
        "max_level_score"
    ].notna()

    medium_high = group[
        "max_level_score"
    ].ge(2)

    high = group[
        "max_level_score"
    ].ge(3)

    reliable_medium_high = group[
        "max_reliable_score"
    ].ge(2)


    tissue_records.append({

        "ensembl_stable_id": ensembl_id,

        "phase55_normal_tissues_recorded":
            int(len(group)),

        "phase55_normal_tissues_interpretable":
            int(measured.sum()),

        "phase55_normal_tissues_medium_high":
            int(medium_high.sum()),

        "phase55_normal_tissues_high":
            int(high.sum()),

        "phase55_normal_tissues_reliable_medium_high":
            int(reliable_medium_high.sum()),

        "phase55_normal_medium_high_tissue_names":
            "; ".join(
                sorted(
                    group.loc[
                        medium_high,
                        "Tissue"
                    ].astype(str).tolist()
                )
            ),

        "phase55_normal_high_tissue_names":
            "; ".join(
                sorted(
                    group.loc[
                        high,
                        "Tissue"
                    ].astype(str).tolist()
                )
            ),

        "phase55_normal_reliable_medium_high_tissue_names":
            "; ".join(
                sorted(
                    group.loc[
                        reliable_medium_high,
                        "Tissue"
                    ].astype(str).tolist()
                )
            )
    })


tissue_summary = pd.DataFrame(
    tissue_records
)


if tissue_summary[
    "ensembl_stable_id"
].duplicated().any():

    raise RuntimeError(
        "Duplicate HPA gene summaries."
    )


print(
    "\nGenes with normal-tissue summaries:",
    len(tissue_summary)
)


# ------------------------------------------------------------
# 9. MERGE NORMAL-TISSUE AUDIT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("INTEGRATING ALL-TISSUE CONTEXT")
print("-" * 65)


annotated = source.merge(

    tissue_summary,

    on="ensembl_stable_id",

    how="left",

    validate="one_to_one"

)


if len(annotated) != 948:

    raise RuntimeError(
        "Gene count changed after HPA integration."
    )


if not annotated[
    source.columns
].equals(source):

    raise RuntimeError(
        "Existing research evidence changed."
    )


def parse_bool(series):

    values = (
        series.astype("string")
        .str.strip()
        .str.lower()
    )

    accepted = {
        "true", "false", "1", "0",
        "", "nan", "none", "<na>"
    }

    unexpected = (
        values.notna()
        & ~values.isin(list(accepted))
    )

    if unexpected.any():

        raise RuntimeError(
            "Unexpected boolean values: "
            f"{values[unexpected].unique().tolist()}"
        )

    return (
        values.isin(["true", "1"])
        .fillna(False)
        .astype(bool)
    )


ihc_available = parse_bool(
    annotated["phase5_normal_ihc_available"]
)


tissue_available = (
    annotated[
        "phase55_normal_tissues_recorded"
    ].notna()
)


mismatch = (
    ihc_available != tissue_available
)


print(
    "\nGenes with canonical normal-IHC evidence:",
    int(ihc_available.sum())
)

print(
    "Genes with tissue-profile records:",
    int(tissue_available.sum())
)

print(
    "Availability disagreements:",
    int(mismatch.sum())
)


if mismatch.any():

    print(
        "\nWARNING: Some IHC availability fields "
        "disagree with the tissue-profile inventory."
    )

    print(
        "These records require review before "
        "making tissue-selectivity claims."
    )


print(
    "\nMissing tissue profiles remain missing, "
    "not zero expression."
)


# ------------------------------------------------------------
# 10. PREPARE EXPERIMENTAL VALIDATION QUESTIONS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENERATING CANDIDATE VALIDATION PRIORITIES")
print("-" * 65)


def validation_questions(row):

    questions = []

    lane = str(
        row["phase5_followup_lane"]
    )

    protein_state = str(
        row["phase5_protein_evidence_state"]
    )

    normal_state = str(
        row["phase5_normal_tissue_review_state"]
    )


    if lane == STROMAL_LANE:

        questions.append(
            "Verify expression in tumor-associated "
            "fibroblasts versus malignant cells"
        )

    elif lane == SURFACE_LANE:

        questions.append(
            "Confirm malignant-cell surface expression "
            "using single-cell/spatial and protein assays"
        )

        questions.append(
            "Evaluate extracellular epitope accessibility"
        )

    elif lane == MEMBRANE_LANE:

        questions.append(
            "Verify plasma-membrane residency "
            "and extracellular topology"
        )

    elif lane == EXTRACELLULAR_LANE:

        questions.append(
            "Establish extracellular localization "
            "and appropriate targeting mechanism"
        )

    else:

        questions.append(
            "Clarify protein localization "
            "before antibody-target consideration"
        )


    if protein_state == (
        "Sparse paired-protein measurements"
    ):

        questions.append(
            "Acquire adequately covered tumor/normal "
            "protein measurements"
        )

    elif protein_state in [
        "Paired-protein coverage unavailable",
        "No complete MSS paired-protein measurement"
    ]:

        questions.append(
            "Obtain independent tumor/normal "
            "protein-level evidence"
        )

    elif protein_state.startswith(
        "Adequately covered"
    ):

        questions.append(
            "Review nonqualifying protein-enrichment "
            "results and assay detectability"
        )


    if normal_state == (
        "Normal-tissue IHC unavailable"
    ):

        questions.append(
            "Acquire independent normal-tissue "
            "expression and selectivity data"
        )

    else:

        n_high = row[
            "phase55_normal_tissues_high"
        ]

        n_medium_high = row[
            "phase55_normal_tissues_medium_high"
        ]


        if (
            pd.notna(n_high)
            and n_high > 0
        ):

            questions.append(
                "Review cell-type-specific high staining "
                "in healthy tissues"
            )

        elif (
            pd.notna(n_medium_high)
            and n_medium_high > 0
        ):

            questions.append(
                "Review medium staining "
                "in healthy tissues"
            )


    if row["gene_name"] == "TDGF1":

        questions.append(
            "Verify CRIPTO membrane presentation, "
            "shedding and normal-tissue selectivity"
        )


    if row["gene_name"] == "FAP":

        questions.append(
            "Investigate tumor-stroma expression "
            "and stromal targeting strategy"
        )


    return "; ".join(questions)


annotated["phase55_validation_questions"] = (
    annotated.apply(
        validation_questions,
        axis=1
    )
)


annotated["phase55_cell_origin_status"] = np.where(

    annotated["gene_name"].eq("FAP"),

    "Stromal-associated working hypothesis; "
    "not verified in this dataset",

    "Cellular origin unresolved from bulk RNA/proteomics"
)


print(
    "\nCandidate validation questions prepared:",
    len(annotated)
)


# ------------------------------------------------------------
# 11. BUILD EVIDENCE DOSSIER PANEL
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("BUILDING CANDIDATE EVIDENCE DOSSIERS")
print("-" * 65)


# Include the union of all three surface
# top-20 lists, plus leading biological genes
# and representative alternative-lane candidates.
#
# This is a review panel, not a final shortlist.


panel_ids = set()


for column in SCENARIO_COLUMNS.values():

    panel_ids.update(
        surface.loc[
            surface[column].le(20),
            "gene_id"
        ].tolist()
    )


panel_ids.update(

    source.loc[
        source["gene_name"].isin(
            REVIEW_GENES
        ),
        "gene_id"
    ].tolist()

)


for lane in [
    MEMBRANE_LANE,
    EXTRACELLULAR_LANE
]:

    group = source[
        source["phase5_followup_lane"].eq(lane)
    ].copy()

    group = group.sort_values(
        [
            "phase5_median_scenario_rank",
            "gene_id"
        ]
    )

    panel_ids.update(
        group.head(5)["gene_id"].tolist()
    )


panel = annotated[
    annotated["gene_id"].isin(panel_ids)
].copy()


panel = panel.sort_values(
    [
        "phase5_followup_lane",
        "phase5_median_scenario_rank",
        "gene_id"
    ]
)


print("\nEvidence dossiers created:", len(panel))

print(
    "These candidates are selected "
    "for detailed review, not declared "
    "therapeutically superior."
)


# ------------------------------------------------------------
# 12. REVIEW SELECTED BIOLOGICAL CANDIDATES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LEADING BIOLOGICAL CANDIDATE DOSSIERS")
print("-" * 65)


leading = panel[
    panel["gene_name"].isin(REVIEW_GENES)
].copy()


leading["display_order"] = (
    leading["gene_name"]
    .map({
        gene: i
        for i, gene in enumerate(
            REVIEW_GENES
        )
    })
)


leading = leading.sort_values(
    "display_order"
)


display_columns = [
    "gene_name",
    "phase5_followup_lane",
    "phase5_protein_evidence_state",
    "phase55_normal_tissues_medium_high",
    "phase55_normal_tissues_high",
    "phase55_normal_tissues_reliable_medium_high",
    "phase5_rna_conservative_log2fc"
]


print("\n")

print(
    leading[
        display_columns
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)


print(
    "\nNormal-tissue counts are descriptive "
    "and do not establish clinical safety."
)


# ------------------------------------------------------------
# 13. PREPARE NORMAL-TISSUE DETAIL TABLE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING TISSUE-LEVEL DOSSIER EVIDENCE")
print("-" * 65)


panel_ensembl_ids = set(
    panel["ensembl_stable_id"]
)


panel_tissues = tissues[
    tissues["Gene"].isin(
        panel_ensembl_ids
    )
].copy()


panel_tissues = panel_tissues.merge(

    panel[
        [
            "ensembl_stable_id",
            "gene_name",
            "phase5_followup_lane"
        ]
    ],

    left_on="Gene",

    right_on="ensembl_stable_id",

    how="left",

    validate="many_to_one"

)


print(
    "\nTissue-level dossier records:",
    len(panel_tissues)
)


# ------------------------------------------------------------
# 14. VALIDATE ORIGINAL RESEARCH EVIDENCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING SOURCE DATA PRESERVATION")
print("-" * 65)


if len(annotated) != 948:

    raise RuntimeError(
        "Original candidate universe changed."
    )


if annotated["gene_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate genes in evidence audit."
    )


if not annotated[
    source.columns
].equals(source):

    raise RuntimeError(
        "Original ranking evidence was modified."
    )


print("\nOriginal 948 candidates: PRESERVED")

print("Original ranking columns: UNCHANGED")

print("No scoring weights introduced.")


# ------------------------------------------------------------
# 15. SAVE RANKING ROBUSTNESS RESULTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING ROBUSTNESS AND DOSSIER TABLES")
print("-" * 65)


rank_correlations.to_csv(
    RESULTS /
    "surfacetarget_phase5_scenario_rank_correlations.csv"
)


overlap.to_csv(
    RESULTS /
    "surfacetarget_phase5_topn_overlap_audit.csv",
    index=False
)


consensus.to_csv(
    RESULTS /
    "surfacetarget_phase5_topn_consensus_audit.csv",
    index=False
)


three_way.to_csv(
    RESULTS /
    "surfacetarget_phase5_three_scenario_top20.csv",
    index=False
)


annotated.to_csv(
    RESULTS /
    "surfacetarget_phase5_all_tissue_evidence_audit.csv",
    index=False
)


panel.to_csv(
    RESULTS /
    "surfacetarget_phase5_candidate_evidence_dossiers.csv",
    index=False
)


leading.to_csv(
    RESULTS /
    "surfacetarget_phase5_leading_candidate_dossiers.csv",
    index=False
)


panel_tissues.to_csv(
    RESULTS /
    "surfacetarget_phase5_dossier_tissue_details.csv",
    index=False
)


print("\nAll ranking diagnostics: SAVED")

print("All candidate evidence dossiers: SAVED")


# ------------------------------------------------------------
# 16. SAVE PROVENANCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING RESEARCH PROVENANCE")
print("-" * 65)


provenance_output = {

    "project": "SurfaceTarget-CRC",

    "patch": "5.5",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "ranking_source": str(RANKING_PATH),

    "ranking_source_sha256": actual_hash,

    "hpa_tissue_source": str(TISSUE_PATH),

    "hpa_tissue_sha256":
        sha256_file(TISSUE_PATH),

    "candidate_count": 948,

    "surface_review_candidates": 111,

    "ranking_scenarios": list(
        SCENARIO_COLUMNS.keys()
    ),

    "top_n_sensitivity": TOP_N_VALUES,

    "three_scenario_top20_count":
        int(len(three_way)),

    "evidence_dossier_count":
        int(len(panel)),

    "ranking_robustness_policy": (
        "Distinguish agreement under two "
        "scenarios from agreement under "
        "all three scenarios"
    ),

    "normal_tissue_policy": (
        "Inspect all available HPA tissue "
        "profiles, preserving missingness "
        "and reliability information"
    ),

    "malignant_cell_origin_inferred": False,

    "clinical_safety_inferred": False,

    "therapeutic_efficacy_scores": False,

    "previous_rankings_modified": False,

    "candidate_exclusions": 0,

    "limitations": [

        "Ranking scenarios are not independent "
        "experimental validation studies.",

        "Agreement under correlated scenarios "
        "may overstate apparent rank stability.",

        "HPA tissue-level maximum staining "
        "does not resolve all cell types.",

        "Normal IHC evidence does not "
        "establish clinical toxicity or safety.",

        "Bulk tumor enrichment does not "
        "identify malignant-cell expression.",

        "Protein enrichment does not "
        "demonstrate antibody accessibility.",

        "CRISPR dependency does not directly "
        "measure antibody efficacy.",

        "No candidate has been "
        "experimentally validated."

    ]
}


with open(
    META /
    "surfacetarget_phase5_robustness_provenance.json",
    "w"
) as handle:

    json.dump(
        provenance_output,
        handle,
        indent=4
    )


print("\nResearch provenance saved.")


# ------------------------------------------------------------
# 17. CREATE PHASE 5 CHECKPOINT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING PHASE 5 CHECKPOINT")
print("-" * 65)


timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S"
)


checkpoint = (
    CHECKPOINTS /
    f"SurfaceTarget_CRC_Phase5_Patch55_{timestamp}.zip"
)


backup_folders = [
    "data/metadata",
    "data/processed",
    "data/raw/cptac_colon",
    "data/raw/hpa",
    "data/raw/uniprot",
    "data/raw/hgnc",
    "results/tables",
    "results/figures",
    "src",
    "reports",
    "logs"
]


with ZipFile(
    checkpoint,
    "w",
    ZIP_DEFLATED,
    compresslevel=3
) as archive:

    for folder in backup_folders:

        folder_path = ROOT / folder

        if not folder_path.exists():
            continue

        for file_path in folder_path.rglob("*"):

            if file_path.is_file():

                archive.write(
                    file_path,
                    arcname=str(
                        file_path.relative_to(ROOT)
                    )
                )


checkpoint_size = (
    checkpoint.stat().st_size / 1024**2
)


print("\nCheckpoint created successfully.")

print(
    "Checkpoint size:",
    f"{checkpoint_size:.2f} MiB"
)

print("Checkpoint location:")
print(checkpoint)


# ------------------------------------------------------------
# 18. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 5.5 SUMMARY")
print("=" * 65)


print("\nResearch candidates:", len(annotated))

print("Surface-review candidates:", len(surface))

print("Ranking scenarios:", len(SCENARIO_COLUMNS))

print(
    "Three-scenario top-20 agreement:",
    len(three_way)
)

print(
    "Candidate evidence dossiers:",
    len(panel)
)

print(
    "HPA normal tissues reviewed:",
    tissues["Tissue"].nunique()
)


print("\nOutputs:")

print("1. Scenario rank-correlation audit")

print("2. Top-N scenario overlap analysis")

print("3. Two-versus-three scenario consensus")

print("4. Three-scenario top-20 candidate table")

print("5. Full normal-tissue evidence audit")

print("6. Candidate evidence dossiers")

print("7. Leading candidate dossiers")

print("8. Detailed HPA tissue profiles")

print("9. Research provenance")


print("\nOriginal research evidence: PRESERVED")

print("No ranking weights added.")

print("No clinical safety claims.")

print("No candidates excluded.")

print("No experimental validation claimed.")


print("\nCheckpoint: CREATED")

print(
    "Checkpoint size:",
    f"{checkpoint_size:.2f} MiB"
)


print("\nPATCH 5.5 COMPLETE")

print("=" * 65)